In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
import tensorflow as tf

2023-07-01 15:20:41.772219: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-01 15:20:41.811240: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-01 15:20:42.544025: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 50,
    "batch_size": 64,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
}

In [4]:
def get_train_data():
    return np.load("../../../data/CIFAR10/train_data.npy"), np.load(
        "../../../data/CIFAR10/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/CIFAR10/test_data.npy"), np.load(
        "../../../data/CIFAR10/test_labels.npy"
    )

In [5]:
def create_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
    model.add(Dropout(0.2))
    model.add(Dense(10, activation='softmax'))
    return model

In [6]:
X_train, y_train = get_train_data()
print(X_train.shape)

(50000, 32, 32, 3)


In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model)

Rain is initialized
Provisioner created successfully
divider is running


In [9]:
# rain.setup_vms()

In [10]:
# rain.delete_vms()

In [11]:
model = rain.train(X_train, y_train, strategy='sync')

divider received: Success receiving the number of workers
divider is sending data to the coordinator
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
Starting iteration 1/3
sending file:  ../../../Divider/divider/data/1.pkl
divider is sending information file to the coordinator
 divider received: File received successfully
divider begins the iteration
filepath is: ../../../Divider/divider/data/1_1_trained.pkl
filepath is: ../../../Divider/divider/data/2_1_trained.pkl
filepath is: ../../../Divider/divider/data/3_1_trained.pkl
 divider received: one loop is done
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)       

In [12]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

157/157 [==============================] - 11s 48ms/step - loss: 0.6856 - accuracy: 0.7876

Test accuracy: 78.8%
